# Blurify AI - ML Engineering Pipeline

A modular pipeline for developing and testing PII detection and image processing logic, matching the production backend.

### Key Features of this Pipeline:
1. **Word-Level Precision**: Uses character offsets to blur only the PII part of a sentence.
2. **Dynamic Linking**: Directly imports `NERProcessor` and `apply_blur` from the backend core logic.
3. **Alphanumeric ID Support**: Handles patterns like `EMP-2024-0892` using advanced regex and context triggers.

In [ ]:
import sys
import os
import cv2
import easyocr
from PIL import Image
import matplotlib.pyplot as plt
import json
import numpy as np

# Add backend to path
backend_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'backend'))
if backend_path not in sys.path:
    sys.path.append(backend_path)

from app.core.ner import NERProcessor
from app.core.blur import apply_blur

# 1. Initialize Components (Same as backend)
print("Initializing EasyOCR and NERProcessor (IndoBERT)...")
reader = easyocr.Reader(['id', 'en'], gpu=False)
ner = NERProcessor()
print("Initialization complete.")

# 2. Simulation of backend /image/process logic with Word-Level Precision
def simulate_backend_pipeline(image_path):
    if not os.path.exists(image_path):
        print(f"Error: {image_path} not found.")
        return
    
    print(f"\nProcessing image: {image_path}")
    
    # OCR Stage
    results = reader.readtext(image_path)
    detected_entities = []
    
    for (bbox, text, prob) in results:
        # Use the granular detection logic
        pii_matches = ner.detect_pii_granular(text)
        
        if pii_matches:
            x_min = int(min([p[0] for p in bbox]))
            y_min = int(min([p[1] for p in bbox]))
            x_max = int(max([p[0] for p in bbox]))
            y_max = int(max([p[1] for p in bbox]))
            w_full, h_full = x_max - x_min, y_max - y_min
            
            text_len = len(text)
            if text_len == 0: continue

            for match in pii_matches:
                # Calculate granular bounding box for the specific PII word
                # This logic replicates the API precisely
                char_width = w_full / text_len
                
                x_start = x_min + int(match['start'] * char_width)
                x_end = x_min + int(match['end'] * char_width)
                
                # Clamp to original box
                x_start = max(x_min, x_start)
                x_end = min(x_max, x_end)
                sub_w = x_end - x_start
                
                entity = {
                    "text": match['text'],
                    "label": match['label'],
                    "bbox": [x_start, y_min, sub_w, h_full],
                    "confidence": float(prob)
                }
                detected_entities.append(entity)
                print(f"Found {match['label']}: '{match['text']}' in segment '{text}'")

    # Blur Stage (Simulation of /finalize)
    if detected_entities:
        print(f"\n--- JSON result for Frontend (Precise BBoxes) ---")
        print(json.dumps(detected_entities, indent=2))
        
        output_path = f"test_result_{os.path.basename(image_path)}"
        bboxes = [e['bbox'] for e in detected_entities]
        apply_blur(image_path, output_path, bboxes)
        
        # Display Results
        fig, ax = plt.subplots(1, 2, figsize=(20, 10))
        
        # Original with Precise Purple Bounding Boxes
        orig_img_display = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
        for ent in detected_entities:
            x, y, w, h = ent['bbox']
            cv2.rectangle(orig_img_display, (x, y), (x + w, y + h), (128, 0, 128), 3)
            
            label_text = ent['label']
            (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(orig_img_display, (x, y - text_h - 10), (x + text_w, y), (128, 0, 128), -1)
            cv2.putText(orig_img_display, label_text, (x, y - 5), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            
        ax[0].imshow(orig_img_display)
        ax[0].set_title(f"Precise Segmentation (Purple Overlay)\n{os.path.basename(image_path)}")
        
        # Final Blurred Result
        ax[1].imshow(cv2.cvtColor(cv2.imread(output_path), cv2.COLOR_BGR2RGB))
        ax[1].set_title("Final Blurred Output (Word-Level)")
        plt.show()
    else:
        print("No PII detected.")

# 3. Run Tests with Edge Cases
test_cases = [
    "NIK pegawainya EMP-2024-0892",
    "Astaga Aldo juga... dia baru KPR loh",
    "Email lo masih dimas.p@gmail.com?"
]

print("\n--- Testing New Granular NER Logic ---")
for t in test_cases:
    results = ner.detect_pii_granular(t)
    print(f"Text: '{t}'")
    for r in results:
        print(f"  -> {r['label']}: {r['text']} at {r['start']}:{r['end']}")

print("\n--- Running Full Dynamic Pipeline on Test Images ---")
test_images = [
    "../backend/test/Test-Image-PII-4.png" # New Chat Log with NIK & Aldo
]

for sample_image in test_images:
    if os.path.exists(sample_image):
        simulate_backend_pipeline(sample_image)
    else:
        print(f"Error: {sample_image} not found.")